# 3. Run Esri's Building Footprint Extraction – USA (ArcGIS Pro)

Esri's pretrained Mask R-CNN model (the `.dlpk` from ArcGIS Living Atlas) runs through ArcGIS Pro's `DetectObjectsUsingDeepLearning`. It needs ArcGIS Pro 3.x with an Image Analyst license, the deep learning libraries for ArcGIS Pro, a GPU for practical speed, and the downloaded `usa_building_footprints.dlpk` in the repository root.

The model runs in ArcGIS Pro's own Python (`arcgispro-py3`); everything else here runs in the project's environment and calls that Python when it needs the model. This notebook was executed with ArcGIS Pro 3.7 on the synthetic demo. Without ArcGIS Pro it skips the model steps; the import and cleanup at the end still run.

In [1]:
import json
import shutil
from pathlib import Path

from nearmap_buildings.cli import main as _nbf

# Run from the repository root or from notebooks/. Generated files go to notebook_runs/, which git ignores.
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORK = REPO / "notebook_runs" / "03_run_esri"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)


def nbf(*args):
    """Run one nbf command in this process; stop the notebook if it fails."""
    code = _nbf([str(a) for a in args])
    if code:
        raise RuntimeError(f"nbf {args[0]} failed with exit code {code}")

ARCGIS_PYTHON = Path(r"C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe")
DLPK = REPO / "usa_building_footprints.dlpk"
RUN_ESRI = ARCGIS_PYTHON.is_file() and DLPK.is_file()
print("ArcGIS Pro Python:", ARCGIS_PYTHON.is_file(), "| model package:", DLPK.is_file(), "| running Esri:", RUN_ESRI)

ArcGIS Pro Python: True | model package: True | running Esri: True


In [2]:
DEMO = WORK / "demo"
nbf("demo", "--output", DEMO)
sorted(p.name for p in DEMO.iterdir())

nbf("tile", DEMO / "imagery.tif", "--output", WORK / "tiles_256", "--tile-size", 256, "--overlap", 32)
MANIFEST = WORK / "tiles_256" / "manifest.json"

<repo>\notebook_runs\03_run_esri\demo


Prepared 21 tiles: <repo>\notebook_runs\03_run_esri\tiles_256\manifest.json


## One raster: the plan

`run_esri` checks the inputs and returns the exact tool call. The arguments are Esri's defaults except two choices made for comparison: `NO_NMS` keeps overlapping detections from neighbouring chips so the shared cleanup handles them for every method, and the 0.5 threshold keeps low-confidence detections so cut-offs can be tested later.

In [3]:
from nearmap_buildings.esri import run_esri

if DLPK.is_file():
    print(json.dumps(run_esri(DEMO / "imagery.tif", DLPK, WORK / "single.gdb" / "buildings"), indent=2))
else:
    print("Download usa_building_footprints.dlpk from ArcGIS Living Atlas into the repository root.")

{
  "tool": "arcpy.ia.DetectObjectsUsingDeepLearning",
  "in_raster": "<repo>\\notebook_runs\\03_run_esri\\demo\\imagery.tif",
  "out_detected_objects": "<repo>\\notebook_runs\\03_run_esri\\single.gdb\\buildings",
  "in_model_definition": "<repo>\\usa_building_footprints.dlpk",
  "arguments": "padding 128;batch_size 4;threshold 0.5;return_bboxes False",
  "run_nms": "NO_NMS",
  "processing_mode": "PROCESS_AS_MOSAICKED_IMAGE",
  "processorType": "GPU",
  "gpuId": "0",
  "execute": false
}


In ArcGIS Pro's Python, the same call with `--execute` runs the model on one raster:

```bat
"C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe" src\nearmap_buildings\esri.py ^
  --raster imagery.tif --model usa_building_footprints.dlpk --output results.gdb\buildings --execute
```

## A whole city: overlapping chunks

One call over all of Lewisville would run for about nine hours with no way to resume. The run also showed that GPU memory grows with each call inside one ArcGIS process: after 42 chunks the model failed to load with a CUDA out-of-memory error. So the city is split into chunks:

- Each chunk covers only tiles that hold imagery, plus a 512 px overlap on every side.
- Each chunk owns a core that doesn't overlap any other chunk's core.
- Each finished chunk writes a done-marker, so a rerun resumes after the last one.
- ArcGIS Python restarts every 8 chunks, which releases its GPU memory.
- The merge keeps each detection only in the chunk whose core holds its centre, so overlaps are never counted twice.

The chunks are virtual: small VRT files that read windows of the source raster.

In [4]:
import rasterio
from xml.sax.saxutils import escape


def plan_chunks(manifest_path, out_dir, core=10240, overlap=512):
    """Write one VRT per chunk that holds imagery; return the plan with each chunk's core bounds."""
    manifest = json.loads(Path(manifest_path).read_text())
    source = Path(manifest["source"]["path"])
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    with rasterio.open(source) as src:
        W, H, t, srs = src.width, src.height, src.transform, escape(src.crs.to_wkt())
    chunks = []
    for r0 in range(0, H, core):
        for c0 in range(0, W, core):
            r1, c1 = min(r0 + core, H), min(c0 + core, W)
            hits = [x for x in manifest["tiles"] if x["row_off"] < r1 and x["row_off"] + x["height"] > r0
                    and x["col_off"] < c1 and x["col_off"] + x["width"] > c0]
            if not hits:
                continue
            # Shrink to the tiles with imagery, never beyond the core plus the overlap.
            er0 = max(r0 - overlap, 0, min(x["row_off"] for x in hits))
            ec0 = max(c0 - overlap, 0, min(x["col_off"] for x in hits))
            er1 = min(r1 + overlap, H, max(x["row_off"] + x["height"] for x in hits))
            ec1 = min(c1 + overlap, W, max(x["col_off"] + x["width"] for x in hits))
            name, w, h = f"chunk_r{r0 // core:02d}_c{c0 // core:02d}", ec1 - ec0, er1 - er0
            bands = "".join(
                f'<VRTRasterBand dataType="Byte" band="{b}"><NoDataValue>0</NoDataValue><SimpleSource>'
                f'<SourceFilename relativeToVRT="0">{escape(str(source))}</SourceFilename><SourceBand>{b}</SourceBand>'
                f'<SrcRect xOff="{ec0}" yOff="{er0}" xSize="{w}" ySize="{h}"/><DstRect xOff="0" yOff="0" xSize="{w}" ySize="{h}"/>'
                f"</SimpleSource></VRTRasterBand>" for b in (1, 2, 3))
            ox, oy = t @ (ec0, er0)
            (out_dir / f"{name}.vrt").write_text(
                f'<VRTDataset rasterXSize="{w}" rasterYSize="{h}"><SRS>{srs}</SRS>'
                f"<GeoTransform>{ox!r}, {t.a!r}, 0, {oy!r}, 0, {t.e!r}</GeoTransform>{bands}</VRTDataset>")
            (x0, y1), (x1, y0) = t @ (c0, r0), t @ (c1, r1)
            chunks.append({"name": name, "pixels": w * h, "core_bounds": [x0, y0, x1, y1]})
    plan = {"core": core, "overlap": overlap, "chunks": chunks}
    (out_dir / "chunks.json").write_text(json.dumps(plan, indent=1))
    return plan


CHUNKS = WORK / "chunks"
plan = plan_chunks(MANIFEST, CHUNKS, core=512, overlap=64)
[(c["name"], c["pixels"]) for c in plan["chunks"]]

[('chunk_r00_c00', 294912),
 ('chunk_r00_c01', 327680),
 ('chunk_r00_c02', 294912)]

The runner below is written to a file and started with ArcGIS Pro's Python. It takes a budget of chunks per process; the driver loop restarts it until every chunk has a done-marker and stops after three failures in a row.

In [5]:
import subprocess

RUNNER = """
import json, sys, time
from pathlib import Path
chunks, gdb, done, model, src = (Path(p) for p in sys.argv[1:6])
budget = int(sys.argv[6])
sys.path.insert(0, str(src))
import arcpy
from esri import run_esri
done.mkdir(parents=True, exist_ok=True)
if not arcpy.Exists(str(gdb)):
    arcpy.management.CreateFileGDB(str(gdb.parent), gdb.name)
for chunk in json.loads((chunks / "chunks.json").read_text())["chunks"]:
    marker = done / (chunk["name"] + ".json")
    if marker.exists():
        continue
    if budget == 0:
        sys.exit(0)
    budget -= 1
    output = gdb / chunk["name"]
    if arcpy.Exists(str(output)):  # partial output from an interrupted run
        arcpy.management.Delete(str(output))
    start = time.time()
    run_esri(chunks / (chunk["name"] + ".vrt"), model, output, execute=True)
    result = {"chunk": chunk["name"], "seconds": round(time.time() - start, 1),
              "features": int(arcpy.management.GetCount(str(output))[0])}
    marker.write_text(json.dumps(result))
    print(json.dumps(result), flush=True)
print("complete", flush=True)
"""
(WORK / "run_chunks.py").write_text(RUNNER)

GDB, DONE = WORK / "esri_chunks.gdb", WORK / "esri_done"


def run_all_chunks(per_process=8, max_failures=3):
    """Restart ArcGIS Python every `per_process` chunks until every chunk is done."""
    failures = 0
    while True:
        result = subprocess.run([str(ARCGIS_PYTHON), str(WORK / "run_chunks.py"), str(CHUNKS), str(GDB), str(DONE), str(DLPK),
                                 str(REPO / "src" / "nearmap_buildings"), str(per_process)], capture_output=True, text=True)
        print("".join(line + "\n" for line in result.stdout.splitlines() if line.startswith(("{", "complete"))), end="")
        if result.returncode:
            failures += 1
            print(f"process failed ({failures} in a row):", result.stderr.strip().splitlines()[-1:] or "")
            if failures >= max_failures:
                raise RuntimeError("Esri chunk run gave up")
        elif "complete" in result.stdout:
            return
        else:
            failures = 0


if RUN_ESRI:
    run_all_chunks(per_process=2)
else:
    print("Skipped: needs ArcGIS Pro with Image Analyst and the model package.")

{"chunk": "chunk_r00_c00", "seconds": 151.0, "features": 3}
{"chunk": "chunk_r00_c01", "seconds": 76.7, "features": 3}


{"chunk": "chunk_r00_c02", "seconds": 118.6, "features": 3}
complete


## Merge the chunks

Each chunk's output goes through the project's Esri import preset: Confidence is divided by 100, and invalid outlines are kept for cleanup to repair. Empty outlines are dropped; Lewisville had two in 98,307. Then each detection is kept only if its centre lies in its own chunk's core, and cleanup runs at a cut-off: 0.9 was frozen for Lewisville.

In [6]:
import tempfile
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
from nearmap_buildings.import_vectors import import_vectors
from nearmap_buildings.postprocess import clean_polygons, write_layers


def merge_chunks(gdb, plan, done, crs, cutoff=0.9):
    """Keep each detection in the chunk whose core holds its centre, then clean at `cutoff`."""
    parts, counts = [], {"detected": 0, "kept": 0, "empty_outlines": 0}
    with tempfile.TemporaryDirectory() as tmp:
        for chunk in plan["chunks"]:
            if not (Path(done) / (chunk["name"] + ".json")).exists():
                raise RuntimeError(f"{chunk['name']} has not finished")
            source = gpd.read_file(gdb, layer=chunk["name"], fid_as_index=True)
            empty = source.geometry.isna() | source.geometry.is_empty
            counts["empty_outlines"] += int(empty.sum())
            source = source[~empty].rename_axis("source_fid").reset_index()
            if source.empty:
                continue
            staged, target = Path(tmp) / f"{chunk['name']}_in.gpkg", Path(tmp) / f"{chunk['name']}.gpkg"
            source.to_file(staged, layer="detections")
            import_vectors(staged, target, crs, layer="detections", preset="esri", id_field="source_fid")
            frame = gpd.read_file(target)
            mine = frame[frame.geometry.buffer(0).representative_point().within(box(*chunk["core_bounds"]))]
            counts["detected"] += len(frame)
            counts["kept"] += len(mine)
            parts.append(mine.assign(source_id=chunk["name"] + ":" + mine.source_id.astype(str)))
    raw = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=crs)
    result = clean_polygons(raw[raw.score >= cutoff].reset_index(drop=True), crs=crs, min_area=4)
    counts |= {"scores": [round(float(raw.score.min()), 2), round(float(raw.score.max()), 2)],
               "at_cutoff": int((raw.score >= cutoff).sum()), "cleaned": len(result.cleaned)}
    return result, counts


# Esri is much less sure of the flat synthetic roofs than of real ones (about 0.5 to 0.7), so the
# demo keeps everything above the run's own 0.5 threshold. Lewisville used the frozen 0.9.
CUTOFF = 0.5
if RUN_ESRI:
    esri_result, counts = merge_chunks(GDB, plan, DONE, "EPSG:32614", cutoff=CUTOFF)
    write_layers(WORK / "esri_cleaned.gpkg", {"cleaned": esri_result.cleaned, "removed_audit": esri_result.removed}, overwrite=True)
    print(counts)

{'detected': 9, 'kept': 9, 'empty_outlines': 0, 'scores': [0.51, 0.66], 'at_cutoff': 9, 'cleaned': 9}


In [7]:
if RUN_ESRI and len(esri_result.cleaned):
    nbf("evaluate", "--predictions", WORK / "esri_cleaned.gpkg", "--predictions-layer", "cleaned",
        "--reference", DEMO / "reference.gpkg", "--aoi", DEMO / "aoi.geojson", "--metric-crs", "EPSG:32614",
        "--output-json", WORK / "esri_evaluation.json", "--independent-holdout")
    report = json.loads((WORK / "esri_evaluation.json").read_text())
    print({k: report[k] for k in ("true_positives", "false_positives", "false_negatives", "precision", "recall", "f1")})

{"true_positives": 9, "false_positives": 0, "false_negatives": 0, "precision": 1.0, "recall": 1.0, "f1": 1.0}
{'true_positives': 9, 'false_positives': 0, 'false_negatives': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


## Import Esri output produced elsewhere

The city of Lewisville published its own run of the same model, made with `ExtractFeaturesUsingAIModels` and Esri's post-processing. Any such output comes in through the same preset. The demo's Esri-format file imitates a raw run's defects: a duplicate from overlapping chips, a self-intersecting outline, a missed building and a false detection. Cleanup removes the duplicate and repairs the outline. This part needs no ArcGIS.

In [8]:
nbf("import-vectors", "--input", DEMO / "esri_format_predictions.gdb", "--layer", "esri_buildings", "--preset", "esri",
    "--crs", "EPSG:32614", "--output", WORK / "imported.gpkg")
nbf("clean", "--input", WORK / "imported.gpkg", "--output", WORK / "imported_cleaned.gpkg", "--metric-crs", "EPSG:32614")
audit = gpd.read_file(WORK / "imported_cleaned.gpkg", layer="removed_audit")
audit[["source_id", "score", "removed_reason", "duplicate_iou"]]

{
  "output": "<repo>\\notebook_runs\\03_run_esri\\imported.gpkg",
  "features": 10,
  "crs": "EPSG:32614",
  "method": "esri_building_usa",
  "source_id": "source feature ID",
  "score_field": "Confidence",
  "score_scale": 100.0,
  "invalid_polygons_kept": 1
}


{"input_count": 10, "cleaned_count": 9, "removed_count": 1, "duplicate_priority": "complete", "raw_inputs_preserved": true}

,source_id,score,removed_reason,duplicate_iou
0,4,0.91,duplicate_overlap,0.891253


## The Lewisville run

| | Lewisville |
|---|---|
| Chunks | 150 of 10,240 px cores with 512 px overlap; 153 km² of imagery including overlaps |
| Time | 8 h 46 min of model time on the RTX 4050 |
| Outlines | 98,307 raw detections, 83,530 kept by core ownership, 41,751 at ≥ 0.9, 39,840 after cleanup |

Next: [4. Score and compare](04_score_and_compare.ipynb).